<a href="https://colab.research.google.com/github/rathans48/flyrank-ml-internship-starter/blob/main/work/notebooks/w06_validation_audit.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/flyrank-bih/flyrank-ml-internship-starter/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

The paper flags survivorship bias on the tiny 365+ × 361+ cell, which is good discipline — but it doesn't address selection into the "refreshed" group itself. My question: how were refresh candidates chosen? If editors picked pages that already showed some residual demand or recoverability, the 3.2x/57x gap could partly reflect which pages get refreshed, not what refreshing does. A stronger version would compare refresh candidates' trends in the window right before the refresh decision, not just before/after outcomes.

Credit where due — the paper itself discloses that health score is partly constructed from position and impressions, so this importance ranking is circular by design. My question: does citing 43%/32% anywhere outside the appendix (e.g. in the playbook) risk being read as causal, despite the disclosure? A stronger safeguard would report importance only on the residual features (age, word count, days visible) to show what predicts health independent of its own ingredients.

In [1]:
%pip -q install duckdb
import duckdb
import pandas as pd
import numpy as np
from google.colab import userdata

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{userdata.get('HF_TOKEN')}')")

DAILY = "read_parquet('hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet')"
CONTENT = "read_parquet('hf://datasets/FlyRank/internship-warehouse/dim_content.parquet')"

In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


# Finding #4: refresh effect and the unstable 361+ freshness bucket
health_boost = 34.5 / 10.7
impression_boost = 4039 / 71
print(f"Refresh health boost: {health_boost:.2f}x (paper claims 3.2x)")
print(f"Refresh impression boost: {impression_boost:.1f}x (paper claims 57x)")

growing_361plus, declining_361plus = 283, 1
ratio_361plus = growing_361plus / declining_361plus
print(f"361+ growth ratio: {ratio_361plus:.0f}:1 on n={growing_361plus + declining_361plus} — "
      f"paper itself flags this as unstable")

# ML Appendix: feature importance disclosure check
importances = {"avg_position": 43, "impressions": 32, "scroll_depth": 15, "ctr": 8, "clicks": 2}
health_score_inputs = {"impressions", "avg_position", "ctr", "scroll_depth"}  # per paper's own Health Score formula
overlap = health_score_inputs & set(importances.keys())
print(f"Features shared between Health Score formula and top RF importances: {overlap}")
print(f"Share of top-2 importance ({importances['avg_position'] + importances['impressions']}%) "
      f"that comes from Health Score's own ingredients")

Refresh health boost: 3.22x (paper claims 3.2x)
Refresh impression boost: 56.9x (paper claims 57x)
361+ growth ratio: 283:1 on n=284 — paper itself flags this as unstable
Features shared between Health Score formula and top RF importances: {'impressions', 'scroll_depth', 'avg_position', 'ctr'}
Share of top-2 importance (75%) that comes from Health Score's own ingredients


## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

Week 5 already used a client-grouped 80/20 split (44/11 clients, seed 42), the honest choice since content from one client shares hidden structure a row-level split would let the model memorize. Rerunning with a naive random split gives [0.334] vs the grouped [0.371].

In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

df = con.execute(f"""
    SELECT d.content_hash_id, d.client_hash_id,
           SUM(d.gsc_impressions) AS impressions_month,
           AVG(d.gsc_avg_position) AS avg_position,
           SUM(d.gsc_clicks) / NULLIF(SUM(d.gsc_impressions), 0) * 100 AS ctr,
           DATE_DIFF('day', c.content_created_date, DATE '2026-03-31') AS content_age_days
    FROM {DAILY} d JOIN {CONTENT} c USING (content_hash_id, client_hash_id)
    WHERE d.gsc_data_available = TRUE
    GROUP BY d.content_hash_id, d.client_hash_id, c.content_created_date
    ORDER BY d.content_hash_id, d.client_hash_id
""").df()

print("Rows after filtering to gsc_data_available=TRUE:", len(df))
print(df.isna().sum())

df['log_impressions'] = np.log1p(df['impressions_month'])
features = ['log_impressions', 'avg_position', 'ctr', 'content_age_days']
print(df[features].isna().sum())

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.model_selection import GroupShuffleSplit, train_test_split

X_scaled = StandardScaler().fit_transform(df[features])

# BEFORE: random row split — same client can appear in both train and test
Xtr_r, Xte_r = train_test_split(X_scaled, test_size=0.2, random_state=42)
km_r = KMeans(n_clusters=5, random_state=42, n_init=10).fit(Xtr_r)
sil_random = silhouette_score(Xte_r, km_r.predict(Xte_r), sample_size=5000, random_state=42)

# AFTER: client-grouped split — same as Week 5
gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
tr_idx, te_idx = next(gss.split(X_scaled, groups=df['client_hash_id']))
km_g = KMeans(n_clusters=5, random_state=42, n_init=10).fit(X_scaled[tr_idx])
sil_grouped = silhouette_score(X_scaled[te_idx], km_g.predict(X_scaled[te_idx]), sample_size=5000, random_state=42)

print(f"Random split silhouette:  {sil_random:.3f}")
print(f"Grouped split silhouette: {sil_grouped:.3f}")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Rows after filtering to gsc_data_available=TRUE: 176738
content_hash_id      0
client_hash_id       0
impressions_month    0
avg_position         0
ctr                  0
content_age_days     0
dtype: int64
log_impressions     0
avg_position        0
ctr                 0
content_age_days    0
dtype: int64
Random split silhouette:  0.334
Grouped split silhouette: 0.371


## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

No supervised label here, so the label-derived-feature check doesn't apply directly, but the rest of the checklist does. content_hash_id/client_hash_id are used only for grouping, never as features. No product flags or existing-system scores (Week 4's baseline reason codes) feed the clustering — they're comparison-only.


The real find this week was a population-selection leak, not a feature leak: filtering to gsc_data_available = TRUE (Section 2) dropped 46.7% of rows that were zero-filled placeholders, not real zero-traffic days. This wasn't disclosed in Week 5 — the original baseline-vs-KMeans comparison (0.381 vs 0.212) was computed on the unfiltered ~331K rows, meaning roughly half the population had fake zeros silently pulling down impressions and CTR. That's exactly the kind of population-definition choice the leakage skill says to disclose, not hide.


Also worth naming honestly: the random-vs-grouped comparison in Section 2 (0.334 vs 0.371) does not show the classic "grouped split deflates an inflated score" pattern the skill describes for supervised models. K-Means has no label to memorize across a group boundary — client overlap in a random split doesn't let it cheat the way it lets a classifier cheat. The two numbers instead reflect two different test populations (a scattered 20%-of-rows sample vs a coarser 20%-of-clients sample), not a leak being exposed. Flagging this so the comparison isn't mistaken for the classic leakage story it initially looks like.

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

# Confirm no product-flags or label-derived features leaked into X
assert 'content_hash_id' not in features and 'client_hash_id' not in features
print("Feature set:", features)

unique, counts = np.unique(km_g.predict(X_scaled[te_idx]), return_counts=True)
print("Cluster sizes (test):", dict(zip(unique, counts)))

# Population-selection check: did filtering to gsc_data_available=TRUE
# introduce outcome-window information? Confirm it's a coverage flag,
# not a flag derived from clicks/impressions the model then sees.
print(con.execute(f"""
    SELECT gsc_data_available, AVG(gsc_impressions) AS avg_impr, AVG(gsc_clicks) AS avg_clicks
    FROM {DAILY} GROUP BY gsc_data_available
""").df())

Feature set: ['log_impressions', 'avg_position', 'ctr', 'content_age_days']
Cluster sizes (test): {np.int32(0): np.int64(4945), np.int32(1): np.int64(17708), np.int32(2): np.int64(46), np.int32(3): np.int64(10565), np.int32(4): np.int64(5164)}


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

   gsc_data_available   avg_impr  avg_clicks
0               False   0.000000    0.000000
1                True  77.721642    0.227587


## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

Observed: on a client-grouped test split restricted to content-days with real GSC coverage (176,738 rows, excluding zero-filled placeholder rows), K-Means (k=5) measured a silhouette score of 0.371 versus 0.061 for the baseline reason-code grouping (which produced only 2 of its 3 possible codes in this test split). This is a directional signal, not proof of superiority — decision-support for which segmentation is worth investigating further, not a claim that the clusters are stable real-world archetypes.

This supersedes Week 5's original comparison (0.381 vs 0.212), which was computed on an unfiltered population that included ~46.7% of content-days with zero-filled GSC data rather than real "no traffic" days. The corrected comparison shows the same direction, with the baseline's advantage shrinking further under the honest population — the baseline's already-limited signal (missing one of three reason codes in test) degrades more than K-Means's does once placeholder rows are removed.

In [5]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Rebuild Week 4's baseline (reason-code buckets) on the SAME filtered population
# used for this week's grouped-split K-Means, for a like-for-like comparison.

def baseline_reason_code(row):
    stale = row['content_age_days'] >= 365
    visible = row['impressions_month'] >= 500
    # tier-relative ctr_gap needs a position-tier median CTR — compute per tier below
    return stale, visible

# Position tiers, same cut points as the paper/Week 4
df['position_tier'] = pd.cut(
    df['avg_position'],
    bins=[0, 3, 10, 20, 50, np.inf],
    labels=['top_3', 'page_1', 'striking', 'page_3_5', 'deep']
)
tier_median_ctr = df.groupby('position_tier', observed=True)['ctr'].transform('median')
df['ctr_gap'] = df['ctr'] < 0.5 * tier_median_ctr
df['stale'] = df['content_age_days'] >= 365
df['visible'] = df['impressions_month'] >= 500

def reason_code(row):
    if row['stale'] and row['visible'] and row['ctr_gap']:
        return 'stale_visible_ctr_gap'
    elif row['stale'] and row['visible']:
        return 'stale_but_visible'
    else:
        return 'not_flagged'

df['reason_code'] = df.apply(reason_code, axis=1)

# Apply the SAME grouped train/test split indices as this week's K-Means (tr_idx, te_idx)
baseline_test_labels = df.iloc[te_idx]['reason_code']
baseline_test_features = X_scaled[te_idx]

n_groups_baseline = baseline_test_labels.nunique()
sil_baseline_filtered = silhouette_score(
    baseline_test_features,
    baseline_test_labels.astype('category').cat.codes,
    sample_size=5000, random_state=42
)

print(f"Baseline (filtered pop, grouped split): n_groups={n_groups_baseline}, silhouette={sil_baseline_filtered:.3f}")
print(f"K-Means  (filtered pop, grouped split): silhouette={sil_grouped:.3f}")
print(f"\nFor reference — Week 5's original unfiltered numbers: baseline=0.212, K-Means=0.381")

Baseline (filtered pop, grouped split): n_groups=2, silhouette=0.054
K-Means  (filtered pop, grouped split): silhouette=0.371

For reference — Week 5's original unfiltered numbers: baseline=0.212, K-Means=0.381


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.